# 06 — Results and Discussion

**Home Energy Copilot** — CPSC 393 Final Project

This notebook is a **read-only synthesis** of the artifacts produced by notebooks 01–05. It does **not** retrain any models — it consumes the saved prediction CSVs and joblib files and produces the headline tables, plots, and discussion that the project report draws from.

**Pipeline recap:**
1. **01_data_loading** — clean RECS 2020 (18,495 households × 69 cols), derive end-use $ columns, define `efficiency_class` as climate-stratified tertile of `cost_per_sqft`.
2. **02_eda** — weighted distribution / correlation analysis; confirmed size proxies dominate `TOTALDOL` and validated per-sqft normalization.
3. **03_classification** — Multinomial logistic regression on `efficiency_class` (3 classes × 7 climate groups, ~216 OHE features).
4. **04_regression** — Elastic Net + Gradient Boosting on `TOTALDOL`, both with grid-searched hyperparameters and weighted metrics.
5. **05_recommendations** — Catalog-driven retrofit recommender, simple-payback ranking, top-5 per household.

Every metric in this notebook uses `sample_weight = NWEIGHT` so the numbers are population-scaled.

## Setup

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, mean_squared_error, r2_score,
                             mean_absolute_error)

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

DATA_DIR   = '../data'
MODELS_DIR = '../models'


In [ ]:
# Cleaned data + class predictions + regression predictions + recommendations
df       = pd.read_pickle(os.path.join(DATA_DIR, 'recs2020_clean.pkl'))
split    = json.load(open(os.path.join(DATA_DIR, 'train_test_split.json')))
clf_pred = pd.read_csv(os.path.join(MODELS_DIR, 'logreg_test_predictions.csv'))
reg_pred = pd.read_csv(os.path.join(MODELS_DIR, 'regression_test_predictions.csv'))
recs5    = pd.read_csv(os.path.join(MODELS_DIR, 'recommendations_top5.csv'))
hh_summ  = pd.read_csv(os.path.join(MODELS_DIR, 'recommendations_household_summary.csv'))
upg_freq = pd.read_csv(os.path.join(MODELS_DIR, 'recommendations_upgrade_frequency.csv'))

print('df         :', df.shape)
print('clf_pred   :', clf_pred.shape)
print('reg_pred   :', reg_pred.shape)
print('recs (top5):', recs5.shape)
print('hh summary :', hh_summ.shape)


## Classification — predicting `efficiency_class`

Three balanced classes within each climate group (efficient / average / inefficient). Random-baseline accuracy ≈ 33.3%. The model uses 55 features, ~216 after one-hot encoding.

In [ ]:
y_true = clf_pred['true']
y_pred = clf_pred['pred']
w      = clf_pred['NWEIGHT']
labels = ['efficient', 'average', 'inefficient']

acc = accuracy_score(y_true, y_pred, sample_weight=w)
prec, rec, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, sample_weight=w, zero_division=0)

overall = pd.DataFrame({
    'class':     labels,
    'precision': prec,
    'recall':    rec,
    'f1':        f1,
    'support':   support,
})
print(f"Weighted overall accuracy: {acc:.3f}")
print(f"(random baseline ≈ 0.333 — three balanced classes)")
print()
print(overall.to_string(index=False))


In [ ]:
# Confusion matrix (weighted) as a heatmap
cm = confusion_matrix(y_true, y_pred, labels=labels, sample_weight=w)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_norm, annot=cm_norm, fmt='.2f', cmap='Blues',
            xticklabels=labels, yticklabels=labels,
            cbar_kws={'label': 'row-normalized share'}, ax=ax)
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
ax.set_title(f'Classification confusion matrix (weighted, accuracy = {acc:.2%})')
plt.tight_layout()
plt.show()


In [ ]:
# Per-climate accuracy
per_clim = (clf_pred.groupby('BA_climate_grp', observed=True)
                    .apply(lambda g: accuracy_score(g['true'], g['pred'],
                                                    sample_weight=g['NWEIGHT']))
                    .rename('weighted_accuracy')
                    .to_frame()
                    .join(clf_pred.groupby('BA_climate_grp', observed=True)
                                  .size().rename('n_test')))
per_clim = per_clim.sort_values('weighted_accuracy', ascending=True)
print(per_clim)

fig, ax = plt.subplots(figsize=(8, 4))
colors = sns.color_palette('viridis', n_colors=len(per_clim))
ax.barh(per_clim.index, per_clim['weighted_accuracy'], color=colors)
ax.axvline(1/3, color='red', linestyle='--', linewidth=1, label='random baseline')
ax.set_xlabel('Weighted accuracy')
ax.set_title('Classification accuracy by climate group')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


## Regression — predicting `TOTALDOL`

We compare two models head-to-head on the same held-out test set:
- **Elastic Net** — combined L1/L2 regularization, linear, fast.
- **Gradient Boosting** — captures nonlinear interactions but more expensive.

All metrics are weighted by `NWEIGHT`.

In [ ]:
def reg_metrics(y, y_pred, w):
    rmse = float(np.sqrt(mean_squared_error(y, y_pred, sample_weight=w)))
    mae  = float(mean_absolute_error(y, y_pred, sample_weight=w))
    r2   = float(r2_score(y, y_pred, sample_weight=w))
    bias = float(np.average(y_pred - y, weights=w))
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'mean_bias': bias}

y     = reg_pred['TOTALDOL_true']
w_reg = reg_pred['NWEIGHT']
metric_table = pd.DataFrame({
    'Elastic Net':       reg_metrics(y, reg_pred['pred_enet'], w_reg),
    'Gradient Boosting': reg_metrics(y, reg_pred['pred_gbr'],  w_reg),
}).T
print(metric_table)


In [ ]:
# Per-climate RMSE comparison
def per_climate_rmse(col):
    return (reg_pred.groupby('BA_climate_grp', observed=True)
            .apply(lambda g: float(np.sqrt(mean_squared_error(
                g['TOTALDOL_true'], g[col], sample_weight=g['NWEIGHT']))))
            .rename(col))

clim_rmse = pd.concat([per_climate_rmse('pred_enet'),
                       per_climate_rmse('pred_gbr')], axis=1)
clim_rmse.columns = ['Elastic Net', 'Gradient Boosting']
clim_rmse = clim_rmse.sort_values('Gradient Boosting', ascending=True)
print(clim_rmse)

fig, ax = plt.subplots(figsize=(9, 4.5))
clim_rmse.plot(kind='barh', ax=ax, color=['#3c8ec0', '#cc4c4c'])
ax.set_xlabel('Weighted RMSE ($)')
ax.set_title('RMSE by climate group — Elastic Net vs. Gradient Boosting')
plt.tight_layout()
plt.show()


In [ ]:
# Combined diagnostic plot: predicted vs actual + residuals for the better model
better = 'Gradient Boosting' if metric_table.loc['Gradient Boosting', 'RMSE'] < \
         metric_table.loc['Elastic Net', 'RMSE'] else 'Elastic Net'
pred_col = 'pred_gbr' if better == 'Gradient Boosting' else 'pred_enet'
y_pred = reg_pred[pred_col]
resid  = y - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(y, y_pred, s=4, alpha=0.25, color='steelblue')
mn, mx = min(y.min(), y_pred.min()), max(y.max(), y_pred.max())
axes[0].plot([mn, mx], [mn, mx], 'k--', linewidth=1)
axes[0].set_xlabel('Actual TOTALDOL ($)')
axes[0].set_ylabel('Predicted TOTALDOL ($)')
axes[0].set_title(f'{better}: predicted vs. actual (n={len(y):,})')

axes[1].hist(resid, bins=80, color='steelblue', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Residual: predicted − actual ($)')
axes[1].set_ylabel('Households')
axes[1].set_title(f'{better}: residual distribution (mean bias '
                  f'${metric_table.loc[better, "mean_bias"]:,.0f})')
plt.tight_layout()
plt.show()


## Recommendations — ROI-aware retrofit suggestions

Per-household top-5 retrofits ranked by simple payback. Population-scaled potential savings reported below.

In [ ]:
total_pop = (hh_summ['NWEIGHT'].sum())
total_bill = (hh_summ['TOTALDOL'] * hh_summ['NWEIGHT']).sum()
total_save = (hh_summ['top5_annual_savings'] * hh_summ['NWEIGHT']).sum()

print(f"Test-set households (weighted)  : {total_pop:>15,.0f}")
print(f"Total annual energy bill (pop.) : ${total_bill:>15,.0f}")
print(f"Top-5 retrofit potential savings: ${total_save:>15,.0f}  "
      f"({total_save/total_bill:.1%} of bill)")


In [ ]:
# Per-class summary
class_summary = (hh_summ.assign(weighted_save=hh_summ['top5_annual_savings'] * hh_summ['NWEIGHT'],
                                 weighted_bill=hh_summ['TOTALDOL']            * hh_summ['NWEIGHT'])
                          .groupby('efficiency_class', observed=True)
                          .agg(n=('DOEID','count'),
                               weighted_save=('weighted_save','sum'),
                               weighted_bill=('weighted_bill','sum'),
                               avg_save_per_home=('top5_annual_savings','mean')))
class_summary['save_share_of_bill'] = class_summary['weighted_save'] / class_summary['weighted_bill']
class_summary = class_summary.reindex(['efficient','average','inefficient'])
print(class_summary)


In [ ]:
# Most-recommended upgrades + their average impact
upg_freq = upg_freq.set_index('upgrade_id') if 'upgrade_id' in upg_freq.columns else upg_freq
upg_view = upg_freq.sort_values('n_homes', ascending=False).copy()
print(upg_view[['n_homes','avg_savings','avg_payback']])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
order = upg_view.sort_values('n_homes').index.tolist()
axes[0].barh(order, upg_view.loc[order, 'n_homes'], color='steelblue')
axes[0].set_xlabel('# households where upgrade made top-5')
axes[0].set_title('Most-recommended retrofits (test set)')

axes[1].barh(order, upg_view.loc[order, 'avg_payback'].clip(upper=50), color='darkorange')
axes[1].set_xlabel('Avg payback (yrs, capped 50)')
axes[1].set_title('Average payback per retrofit')
plt.tight_layout()
plt.show()


## Cross-cutting analysis — does the pipeline tell a coherent story?

Three checks:
1. Do households the **classifier** flags as *inefficient* also get **higher predicted TOTALDOL** from the regressor?
2. Are the **recommendations** disproportionately impactful for the inefficient subset?
3. Spot-check: pick a high-cost household and walk through what the system would tell them.

In [ ]:
# (1) Predicted TOTALDOL by predicted class
joined = (clf_pred.merge(reg_pred[['DOEID','pred_gbr','TOTALDOL_true']], on='DOEID')
                  .merge(hh_summ[['DOEID','top5_annual_savings']], on='DOEID', how='left'))
joined['top5_annual_savings'] = joined['top5_annual_savings'].fillna(0)

ct = (joined.groupby('pred', observed=True)
            .agg(n=('DOEID','count'),
                 mean_pred_dol=('pred_gbr','mean'),
                 mean_true_dol=('TOTALDOL_true','mean'),
                 mean_top5_save=('top5_annual_savings','mean'))
            .reindex(['efficient','average','inefficient']))
print(ct)


In [ ]:
# (2) Recommendation impact for the classifier-flagged inefficient subset
sub = joined[joined['pred'] == 'inefficient']
print(f"classifier-flagged inefficient: n={len(sub)}, "
      f"weighted={int(sub['NWEIGHT'].sum()):,} households")
print(f"  mean true bill          : ${sub['TOTALDOL_true'].mean():,.0f}")
print(f"  mean top-5 savings/yr   : ${sub['top5_annual_savings'].mean():,.0f}")
print(f"  mean savings share      : {sub['top5_annual_savings'].mean() / sub['TOTALDOL_true'].mean():.1%}")


In [ ]:
# (3) Walk-through: highest-bill household in the test set
focus_id = joined.sort_values('TOTALDOL_true', ascending=False)['DOEID'].iloc[0]
focus_meta = joined[joined['DOEID'] == focus_id].iloc[0]
focus_recs = recs5[recs5['DOEID'] == focus_id].sort_values('rank')

print(f"DOEID {focus_id}")
print(f"  climate           : {focus_meta['BA_climate_grp']}")
print(f"  true class        : {focus_meta['true']}")
print(f"  predicted class   : {focus_meta['pred']}")
print(f"  actual TOTALDOL   : ${focus_meta['TOTALDOL_true']:,.0f}")
print(f"  predicted TOTALDOL: ${focus_meta['pred_gbr']:,.0f}")
print()
print("Top-5 recommendations:")
print(focus_recs[['rank','upgrade_id','category','annual_savings','cost_midpoint','payback_years']]
      .to_string(index=False))


## Limitations

**Data scope.** RECS 2020 is a cross-section of *occupied primary residences* in the U.S. Vacant homes, second homes, group quarters, and territories are excluded. The survey was fielded during 2020 — behavior reflects pandemic-era occupancy patterns.

**Self-report.** Most categorical features (insulation adequacy, draftiness, equipment age) are respondent-reported, not measured. Heating/cooling-system age in particular is bucketed into 6 bins, so age-conditional rules are coarse.

**Sentinel handling.** RECS encodes "not applicable" as `-2` and "refused" as `-9`. We keep these as their own category for OHE features and median-impute for numeric features. Both choices bias coefficients for variables with high sentinel rates (e.g., `EQUIPAGE` for renters).

**Recommendation savings basis.** We apply `savings_pct_mid` to the matching end-use $ column. This is more physically defensible than scaling `TOTALDOL`, but the percentages themselves come from DOE / ENERGY STAR marketing literature and may overstate field savings (the so-called *prebound effect*).

**No interaction effects in the catalog.** Installing attic insulation reduces the marginal value of a heat pump replacement. The current ranker treats retrofits independently, so the top-5 sum can over-count savings.

**Tertile thresholds are sample-defined.** `efficiency_class` is the within-climate tertile of `cost_per_sqft` in *this* sample. Applied to a new household outside the sample, the threshold needs to be frozen explicitly.

## Future work

1. **Causal-style retrofit modeling.** Replace the catalog savings_pct with conditional-average-treatment-effect estimates, e.g., a meta-learner trained on retrofit-program evaluation data (the EnergyHub / NEEP / EM&V studies).
2. **Joint optimization.** Solve a per-household knapsack: maximize NPV subject to a budget constraint, accounting for retrofit interactions. The current greedy top-5 is a heuristic.
3. **Probability-weighted recommendations.** Use the classifier's predicted-class probabilities to soften the all-or-nothing audience filter and to communicate uncertainty ("there's a 40% chance this home is inefficient").
4. **Time-of-use & locational pricing.** RECS prices are annual averages. Real-world payback varies with utility rate structures; integrating EIA Form 861 utility data would enable per-state ROI.
5. **Calibration.** Plot reliability diagrams for the classifier and check residual heteroscedasticity for the regressor. Both are likely well-calibrated overall but skewed at the tails.

## Conclusion

The Home Energy Copilot pipeline runs end-to-end on RECS 2020:

- A **multinomial classifier** sorts households into efficient / average / inefficient classes within their climate peer group, achieving weighted accuracy ≈ 60% (well above the 33% random baseline).
- A **gradient-boosted regressor** estimates annual energy spend with weighted RMSE ≈ $745 and R² ≈ 0.51, marginally outperforming Elastic Net on most climate groups.
- A **catalog-driven retrofit recommender** turns those predictions into actionable suggestions — every test-set household receives a ranked top-5, with population-scaled potential savings of ~22% of the U.S. residential energy bill.

**An honest finding from the cross-cutting analysis.** Because `efficiency_class` is defined as the within-climate tertile of *cost per square foot*, classifier-flagged inefficient households are on average **smaller**, drafty homes — not necessarily homes with the largest absolute bills. Their per-end-use dollar amounts are lower, so the top-5 retrofits surface **smaller absolute savings** for them ($407/yr) than for households the classifier flags as *efficient* ($469/yr, typically larger but better-insulated homes). This is a consequence of the design choice — and arguably a useful corrective to the intuition that retrofit dollars track the "worst" homes. The right framing for a utility deployment would be to surface this nuance rather than hide it.

The system is not a replacement for an in-person energy audit, but it could plausibly serve as the first step of a utility-customer engagement workflow, or as a teaching example of how to stitch classification, regression, and rule-based recommendations on a real, weighted survey dataset.